# Week 8, Day 1: Derivatives: rates of change and the slope of a tuning curve

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w008/d1.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Forward and central differences

Write `forward_diff(f, x, h)` and `central_diff(f, x, h)`. For $f(x) = e^x$ at $x = 1$ (whose derivative is $e$), compute the absolute error of each for every h in `hs` and plot both on log-log axes. Read off how the error scales with h.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def forward_diff(f, x, h):
    return ...

def central_diff(f, x, h):
    return ...

hs = np.array([1, 0.3, 0.1, 0.03, 0.01, 0.003, 0.001])
err_f = [abs(forward_diff(np.exp, 1.0, h) - np.e) for h in hs]
err_c = [abs(central_diff(np.exp, 1.0, h) - np.e) for h in hs]

plt.figure(figsize=(5, 3.2)); plt.loglog(hs, err_f, "o-", label="forward"); plt.loglog(hs, err_c, "o-", label="central")
plt.xlabel("h"); plt.ylabel("absolute error"); plt.legend(frameon=False); plt.show()
print(np.array(err_f).round(6)); print(np.array(err_c).round(8))

In [ ]:
import numpy as np
def test_forward():
    "forward_diff computes (f(x+h) - f(x)) / h"
    assert np.isclose(forward_diff(lambda x: x ** 2, 3.0, 0.1), 6.1)

def test_central():
    "central_diff computes (f(x+h) - f(x-h)) / 2h"
    assert np.isclose(central_diff(lambda x: x ** 2, 3.0, 0.1), 6.0)

def test_errors():
    "err_f and err_c hold the errors for each h, and central is far more accurate at h = 0.01"
    assert len(err_f) == len(hs) == len(err_c)
    i = list(hs).index(0.01)
    assert err_c[i] < err_f[i] / 50

test_forward()
test_central()
test_errors()
print("All checks passed")

### Exercise: Where is the tuning curve steepest?

Write `dtuning(theta)` using the formula above, and check it against a central difference. Then evaluate it on a fine grid of orientations below the peak and store the orientation of the largest slope in `steepest`. Compare it with $\mu - \sigma$.

In [ ]:
b, A, mu, sigma = 5.0, 40.0, 90.0, 25.0
def tuning(theta):
    return b + A * np.exp(-(theta - mu) ** 2 / (2 * sigma ** 2))

def dtuning(theta):
    return ...

grid = np.linspace(0, 90, 9001)
steepest = ...
print("steepest at", round(steepest, 2), "degrees;  mu - sigma =", mu - sigma)
print("check:", dtuning(60.0).round(5), ((tuning(60.0001) - tuning(59.9999)) / 0.0002).round(5))

In [ ]:
import numpy as np
def test_dtuning():
    "dtuning matches a central difference of tuning"
    for th in [40.0, 75.0, 90.0, 130.0]:
        assert np.isclose(dtuning(th), (tuning(th + 1e-4) - tuning(th - 1e-4)) / 2e-4, rtol=1e-5)

def test_steepest():
    "steepest is the orientation below 90 where the slope is largest, mu - sigma"
    assert abs(steepest - 65) <= 0.5

test_dtuning()
test_steepest()
print("All checks passed")

### Exercise: Where does the neuron discriminate best?

Write `fisher(theta, T)` for a 0.5 s window. Evaluate it for orientations 0 to 180, find the orientation where it is largest (`best_theta`) and the approximate threshold at 60° (`threshold_60`). Plot the tuning curve and the Fisher information together.

In [ ]:
def fisher(theta, T):
    return ...

thetas = np.linspace(0, 180, 1801)
I = fisher(thetas, 0.5)
best_theta = ...
threshold_60 = ...

fig, ax = plt.subplots(figsize=(5.5, 3.2)); ax.plot(thetas, tuning(thetas), label="rate f(θ)"); ax.set_xlabel("orientation (deg)"); ax.set_ylabel("Hz")
ax2 = ax.twinx(); ax2.plot(thetas, I, color="C1", label="Fisher information"); ax2.set_ylabel("Fisher information (1/deg²)")
fig.legend(frameon=False, loc="upper left"); plt.show()
print("best orientation", round(best_theta, 1), "  threshold at 60°:", round(threshold_60, 2), "degrees")

In [ ]:
import numpy as np
def test_info():
    "fisher gives T f'(theta)^2 / f(theta)"
    assert np.isclose(fisher(60.0, 0.5), 0.5 * dtuning(60.0) ** 2 / tuning(60.0))

def test_best():
    "the best orientation is on a flank, far from the peak, and the peak gives zero information"
    assert abs(best_theta - 90) > 15
    assert np.isclose(fisher(90.0, 0.5), 0)

def test_threshold():
    "threshold_60 is 1 / sqrt(I) at 60 degrees"
    assert np.isclose(threshold_60, 1 / np.sqrt(fisher(60.0, 0.5)))
    assert 5 < threshold_60 < 10

test_info()
test_best()
test_threshold()
print("All checks passed")